In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from integral_functions.weight_functions import get_weights
from numpy.typing import NDArray
from scipy.special import expit
from scipy.integrate import quad
from sklearn.metrics import mean_absolute_error, mean_squared_error
import statsmodels.api as sm

from regmod.models.binomial import BinomialModel
from regmod.parameter import Parameter
from regmod.variable import Variable
# For more Cython info on the import statement, visit the link: https://stackoverflow.com/questions/7508803/how-do-i-import-function-from-pyx-file-in-python

from integral_functions.simulation.age_distribution import age_distribution
from integral_functions.simulation.death_rate import death_rate_function, expit_death_rate_function, func1, func2, func3
from integral_functions.simulation.sampling import sample_probability_of_death
from integral_functions.simulation.age_intervals import bounded_intervals_generator, int_bounded_intervals_generator
from integral_functions.simulation.integrate_functions import integrate_cov, integrate_denom

In [4]:
low_age = 0
high_age = 95
age_mid = 35

c1 = 0.001
c2 = 1
c3 = 0.0017

In [5]:
# range for each sample group
np.random.seed(0)
num_groups = 1000
age_interval_lb = 20
age_interval_ub = 30
age_ranges = int_bounded_intervals_generator(low_age, high_age, num_groups, age_interval_lb, age_interval_ub)
df = pd.DataFrame(
    dict(
        age_start=age_ranges[0],
        age_end=age_ranges[1],
    )
)

# sample size for each sample group
df["sample_size"] = np.random.randint(low=10, high=100, size=num_groups)


df["obs"] = df.apply(
    lambda row: sample_probability_of_death(
        row.iloc[0],
        row.iloc[1],
        row.iloc[2],
        age_distribution,
        expit_death_rate_function,
        prob_args= {"c1": c1}
    ),
    axis=1,
)

# add covariates
func_list = [func1, func2, func3]
for i, func in enumerate(func_list):
    df["cov" + str(i + 1)] = df.apply(
        lambda row: integrate_cov(
            func=func,
            density=age_distribution,
            age_start=row.iloc[0],
            age_end=row.iloc[1],
            age_mid=age_mid
        ),
        axis=1,
    )

# add denominators
df["denom"] = df.apply(
    lambda row: integrate_denom(
        density=age_distribution,
        age_start=row.iloc[0],
        age_end=row.iloc[1],
        age_mid=age_mid
    ),
    axis=1,
)

# Add outcomes -- the y_i's
df["cov1"] = df["cov1"] / df["denom"]
df["cov2"] = df["cov2"] / df["denom"]
df["cov3"] = df["cov3"] / df["denom"]
df["outcomes"] = expit((c1 * df["cov1"] + c2 * df["cov2"] + c3 * df["cov3"]))

In [6]:
df

,age_start,age_end,sample_size,obs,cov1,cov2,cov3,denom,outcomes
0,44,67,60,0.166667,54.242932,-7.351654,2985.117748,498.300708,0.097730
1,47,67,54,0.092593,56.026660,-7.475428,3171.563412,414.015408,0.116321
2,64,92,94,0.670213,75.299857,-8.666320,5729.612287,283.057393,0.759387
3,9,37,62,0.016129,23.282353,-4.750777,603.666576,1042.469141,0.024097
4,21,42,35,0.000000,31.041881,-5.545434,999.245084,778.734937,0.021548
...,...,...,...,...,...,...,...,...,...
995,50,78,95,0.210526,61.894968,-7.851637,3892.791080,471.226104,0.236494
996,53,79,88,0.352273,64.120884,-7.994647,4165.007628,408.792608,0.299430
997,10,33,39,0.025641,21.827005,-4.616541,518.747051,868.173333,0.023826
998,43,65,11,0.090909,52.872907,-7.258651,2834.863397,496.876193,0.084207
